# 156 — AgentOps y análisis de trayectorias

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**AgentOps** opera trayectorias, no respuestas: cada tarea es un span raíz con spans
hijos por llamada LLM (`gen_ai.*`) y por herramienta (argumentos, resultado, error).

**Cuatro métricas de agente** (siempre distribuciones, no promedios sueltos):

1. **Éxito de tarea** — verificado externamente, nunca auto-reportado.
2. **Pasos por tarea** — p50/p95; colas largas = bucles o tareas fuera de alcance.
3. **Costo por tarea** — tokens y $ acumulados en todos los pasos.
4. **Intervenciones humanas** — la medida de autonomía real.

**Análisis de trayectorias**: clustering de fallos por patrón, puntos de divergencia
entre trayectorias exitosas y fallidas, y replay contra configuración nueva (con
herramientas de efectos reales mockeadas). Regla de oro: instrumentar desde el día uno —
un agente sin trazas es indepurable.


## 🧮 Ejemplo de referencia

1 000 tareas: éxito 84 %, pasos p50/p95 = 4/19, intervenciones 9 %.

```text
cluster A (62 fallos): reintenta crear_reembolso con el MISMO monto — el error
                       "monto>límite" no decía cuál era el límite
cluster B (41 fallos): la tarea exige una herramienta que no existe
```

Arreglos quirúrgicos: enriquecer el mensaje de error (A cae a 8 fallos) y derivar a
humano en el paso 1 las tareas sin herramienta (B). El p95 de pasos baja de 19 a 9.
Nada de esto se veía en la tasa agregada: salió de leer trayectorias.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=156)
show(result)


## Reflexión

1. ¿Por qué el éxito auto-reportado por el agente sobreestima el éxito real, y qué tres formas de verificación externa podrías usar según el tipo de tarea?
2. En el ejemplo, el arreglo del cluster A fue mejorar el mensaje de error de la herramienta, no el prompt: ¿qué te dice eso sobre dónde vive el «comportamiento» de un agente?
3. ¿Qué condiciones debe cumplir un replay de trayectorias para que la comparación entre la configuración vieja y la nueva sea válida?
